In [1]:
import os
import platform
import sqlite3
import pandas as pd
from pathlib import Path
import re

# Colunas do ficheiro (linha 2 do xlsx)
COLUNAS_FICHEIRO = [
    "EXPEDICIÓN", "ENTREGA", "CODEUT", "CODEDT", "BASE",
    "FACTURANTE", "ACTIVIDAD", "TRANSPORTISTA", "COSTEDT", "FLOTA",
    "CLIENTE", "ORIGEN", "ENTREGAR   EN", "DESTINO", "PALETS",
    "VLUS", "PESODT", "REFERENCIA", "USCODE", "USUARIO",
    "NOTAS", "GESTION", "DEPART", "TEMP MERC PED", "TRACTORA",
    "REMOLQUE", "CAMION_TIPO", "CAMION_CAPACIDAD", "PESODTP"
]

# Nomes seguros para SQLite
COLUNAS_SQL = [
    "EXPEDICION", "ENTREGA", "CODEUT", "CODEDT", "BASE",
    "FACTURANTE", "ACTIVIDAD", "TRANSPORTISTA", "COSTEDT", "FLOTA",
    "CLIENTE", "ORIGEN", "ENTREGAR_EN", "DESTINO", "PALETS",
    "VLUS", "PESODT", "REFERENCIA", "USCODE", "USUARIO",
    "NOTAS", "GESTION", "DEPART", "TEMP_MERC_PED", "TRACTORA",
    "REMOLQUE", "CAMION_TIPO", "CAMION_CAPACIDAD", "PESODTP"
]

MAP_COLUNAS = dict(zip(COLUNAS_FICHEIRO, COLUNAS_SQL))

COLUNA_ORIGEM = "ficheiro_origem"
CAMPO_DATA    = "ENTREGA"
CAMPO_DATA_SQL = "ENTREGA"
REGEX_ANO = re.compile(r"^(\d{4})")

In [2]:
if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
    PASTA_FICHEIROS = Path(r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_26")
elif platform.system() == 'Darwin':
    DB_PATH = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_27.db"
    PASTA_FICHEIROS = Path("/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_26")
else:
    DB_PATH = "inform_27.db"
    PASTA_FICHEIROS = Path("inform_26")

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
con = sqlite3.connect(DB_PATH)
con.close()
print(f"✓ BD: {DB_PATH}")

✓ BD: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_27.db


In [3]:
def obter_ano(valor):
    if valor is None:
        return None
    match = REGEX_ANO.match(str(valor).strip())
    return int(match.group(1)) if match and match.group(1) != "0000" else None


def ler_xlsx(caminho):
    """Lê xlsx com header na linha 2."""
    try:
        df = pd.read_excel(caminho, header=1, dtype=str)
        df = df.where(pd.notnull(df), None)

        colunas_presentes = [c for c in COLUNAS_FICHEIRO if c in df.columns]
        colunas_sql_presentes = [MAP_COLUNAS[c] for c in colunas_presentes]

        linhas = [list(row) for row in df[colunas_presentes].itertuples(index=False, name=None)]
        return colunas_sql_presentes, linhas

    except Exception as e:
        print(f"  ERRO ao ler: {e}")
        return None, None


def tabela_ano(ano):
    return f"inform_26_{ano}"


def criar_tabela(con, ano):
    tabela = tabela_ano(ano)
    colunas_sql_def = ", ".join(f'"{c}" TEXT' for c in COLUNAS_SQL)

    con.execute(f'''
        CREATE TABLE IF NOT EXISTS "{tabela}" (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            {colunas_sql_def},
            "{COLUNA_ORIGEM}" TEXT
        )
    ''')

    con.execute(
        f'CREATE UNIQUE INDEX IF NOT EXISTS "idx_{tabela}_chave" '
        f'ON "{tabela}" ("CODEUT", "CODEDT", "ENTREGA")'
    )
    return tabela


def inserir_linhas(con, ano, colunas_sql_lidas, linhas, nome_ficheiro):
    if not linhas:
        return 0, 0

    tabela = criar_tabela(con, ano)

    # Remapear para ordem fixa de COLUNAS_SQL
    idx_map = {c: i for i, c in enumerate(colunas_sql_lidas)}
    batch = []
    for linha in linhas:
        valores = [linha[idx_map[c]] if c in idx_map else None for c in COLUNAS_SQL]
        valores.append(nome_ficheiro)
        batch.append(valores)

    colunas_str = ", ".join(f'"{c}"' for c in COLUNAS_SQL)
    placeholders = ", ".join("?" for _ in range(len(COLUNAS_SQL) + 1))
    sql = f'INSERT OR IGNORE INTO "{tabela}" ({colunas_str}, "{COLUNA_ORIGEM}") VALUES ({placeholders})'

    antes = con.total_changes
    con.executemany(sql, batch)
    inseridos = con.total_changes - antes

    return inseridos, len(batch) - inseridos

In [4]:
ficheiros = sorted(PASTA_FICHEIROS.rglob("*.xlsx"))

if not ficheiros:
    print(f"⚠️  Nenhum ficheiro .xlsx em: {PASTA_FICHEIROS}")
else:
    con = sqlite3.connect(DB_PATH)
    con.execute("PRAGMA synchronous = OFF")
    con.execute("PRAGMA journal_mode = WAL")

    totais = {"ficheiros": 0, "linhas": 0, "inseridos": {}, "duplicados": 0, "sem_data": 0, "erros": 0}

    for numero, caminho in enumerate(ficheiros, 1):
        try:
            colunas_sql_lidas, linhas = ler_xlsx(caminho)

            if colunas_sql_lidas is None or not linhas:
                print(f"[{numero}/{len(ficheiros)}] {caminho.name} — sem dados")
                totais["erros"] += 1
                continue

            if CAMPO_DATA_SQL not in colunas_sql_lidas:
                print(f"[{numero}/{len(ficheiros)}] {caminho.name} — sem {CAMPO_DATA_SQL}")
                totais["erros"] += 1
                continue

            idx_data = colunas_sql_lidas.index(CAMPO_DATA_SQL)
            linhas_por_ano = {}
            sem_data = 0

            # Filtrar linhas vazias e agrupar por ano
            for linha in linhas:
                if all(v is None or str(v).strip() == "" for v in linha):
                    continue
                ano = obter_ano(linha[idx_data])
                if ano is None:
                    sem_data += 1
                    continue
                linhas_por_ano.setdefault(ano, []).append(linha)

            novos_ficheiro, duplicados_ficheiro = 0, 0

            for ano, batch in sorted(linhas_por_ano.items()):
                inseridos, duplicados = inserir_linhas(con, ano, colunas_sql_lidas, batch, caminho.name)
                totais["inseridos"][ano] = totais["inseridos"].get(ano, 0) + inseridos
                novos_ficheiro += inseridos
                duplicados_ficheiro += duplicados

            con.commit()

            totais["ficheiros"] += 1
            totais["linhas"] += len(linhas)
            totais["duplicados"] += duplicados_ficheiro
            totais["sem_data"] += sem_data

            print(
                f"[{numero}/{len(ficheiros)}] {caminho.name} | "
                f"+{novos_ficheiro:,} novos | "
                f"{duplicados_ficheiro:,} duplicados | "
                f"{sem_data:,} sem data"
            )

        except Exception as erro:
            con.rollback()
            totais["erros"] += 1
            print(f"[{numero}/{len(ficheiros)}] {caminho.name} — ERRO: {erro}")

    con.close()

    print("\n--- RESUMO ---")
    print(
        f"Ficheiros: {totais['ficheiros']} | "
        f"Linhas lidas: {totais['linhas']:,} | "
        f"Duplicadas: {totais['duplicados']:,} | "
        f"Sem data: {totais['sem_data']:,} | "
        f"Erros: {totais['erros']}"
    )
    for ano, qtd in sorted(totais["inseridos"].items()):
        print(f"Linhas novas em {ano}: {qtd:,}")

⚠️  Nenhum ficheiro .xlsx em: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_26
